# small\_fable — full pipeline on Colab (GPU) → Hugging Face

Runs the whole thing end-to-end on a free Colab T4 and **pushes the checkpoints to the Hugging Face Hub**:

1. **Stage 1 — SFT**: joint planner + executor. Held-out validation every epoch.
2. **Stage 2a — Rollouts**: cache G hot samples per prompt.
3. **Stage 2b — Off-policy GRPO**: trains from the cache; held-out reward every inner epoch.
4. **Compare** + **head-to-head** on a hard out-of-distribution prompt.

**Checkpoints → Hugging Face.** Each stage's result is pushed to your HF repo, and an **hourly
autosave** overwrites it in the background — so if Colab recycles (it kills the runtime after ~4h)
you keep the latest finished artifacts. Load them later with `inference_colab.ipynb`.

> **Before running:** `Runtime → Change runtime type → T4 GPU`. You also need a **Hugging Face token**
> with *write* access — add it as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar,
> *Notebook access* ON). Get one at https://huggingface.co/settings/tokens . Then `Runtime → Run all`.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone the repo & install deps

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt huggingface_hub
print('\nReady.')


## 2 · Hugging Face setup + autosave
Logs in with your `HF_TOKEN` secret, creates a **public** model repo `<your-username>/small_fable-planner`,
and arms an **hourly background autosave**. `push_ckpts(reason)` uploads whatever checkpoints exist,
overwriting the same paths in the repo.

In [ ]:
import os, threading, time
from huggingface_hub import HfApi, create_repo

# token from Colab secret 'HF_TOKEN' (preferred), else interactive login
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import notebook_login; notebook_login()

api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']
HF_REPO = f'{HF_USER}/small_fable-planner'
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False, token=HF_TOKEN)
print('HF repo:', f'https://huggingface.co/{HF_REPO}')
print('>>> Use this HF_REPO in the inference notebook:', HF_REPO)

CKPT_DIRS = ['joint_ckpt', 'rl_ckpt']
EXTRA_FILES = ['rl_rollouts.jsonl']

def push_ckpts(reason='update'):
    pushed = []
    for sub in CKPT_DIRS:
        if os.path.isdir(sub):
            api.upload_folder(folder_path=sub, path_in_repo=sub, repo_id=HF_REPO,
                              commit_message=f'{reason}: {sub}')
            pushed.append(sub)
    for fn in EXTRA_FILES:
        if os.path.exists(fn):
            api.upload_file(path_or_fileobj=fn, path_in_repo=fn, repo_id=HF_REPO,
                            commit_message=f'{reason}: {fn}')
            pushed.append(fn)
    print(f'[hf] {reason}: pushed {pushed or "(nothing yet)"} -> https://huggingface.co/{HF_REPO}')
    return pushed

# hourly autosave: Colab recycles ~4h, so overwrite the latest finished artifacts every 60 min.
_autosave_stop = False
def _autosave_loop():
    while not _autosave_stop:
        for _ in range(60):
            if _autosave_stop: return
            time.sleep(60)
        try: push_ckpts('hourly-autosave')
        except Exception as e: print('[hf] autosave skipped:', e)
threading.Thread(target=_autosave_loop, daemon=True).start()
print('hourly autosave armed (background)')


## 3 · Stage 1 — Joint SFT
Watch the `held {...}` line each epoch (held-out validation): `plan_ce`/`resp_ce` fall, `ablation_gap`
stays positive and grows (the plan is load-bearing). _~15–30 min on a T4._

In [ ]:
!python -u train_sft.py \
    --data dataset/sft_100.jsonl --train 70 --held 30 \
    --epochs 6 --lr 2e-5 --bs 4 --max_resp 64 \
    --probe 12 --out joint_ckpt --device cuda


In [ ]:
push_ckpts('after-SFT')   # final SFT checkpoint -> HF


## 4 · Stage 2a — Offline rollouts
Cache `G=8` hot samples per prompt. _~10–20 min on a T4._

In [ ]:
!python -u rollout_offline.py \
    --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --train 80 --group 8 --temp 1.5 --top_p 0.98 --max_resp 64 \
    --out rl_rollouts.jsonl --device cuda


In [ ]:
push_ckpts('after-rollouts')   # cache the rollouts too


## 5 · Stage 2b — Off-policy GRPO
Watch `held_reward=` each inner epoch (held-out validation moving under RL). Final `|ΔL2| > 0` proves
the adapter actually moved (RL ≠ SFT).

In [ ]:
!python -u train_grpo_offline.py \
    --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data dataset/sft_100.jsonl \
    --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 \
    --beta_plan 1.0 --beta_ce 0.1 --held 16 --device cuda


In [ ]:
push_ckpts('final')                    # final SFT+RL checkpoint -> HF
_autosave_stop = True                  # stop the hourly thread; we're done
print('All checkpoints on HF:', f'https://huggingface.co/{HF_REPO}/tree/main')


## 6 · Compare — base vs SFT vs SFT+RL
`--sample` (temp 0.7) is required to see small RL effects.

In [ ]:
!python -u compare.py \
    --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt \
    --data dataset/sft_100.jsonl --train 80 --held 20 \
    --sample --device cuda


## 7 · Head-to-head on a brand-new hard prompt
A multi-step prompt that appears **nowhere** in the data. (For a standalone inference run, use
`inference_colab.ipynb`, which loads these checkpoints straight from Hugging Face.)

In [ ]:
import torch
from model_joint import JointModel, decode_plan
HARD_PROMPT = ('A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
               'but each night it slides back 3 meters. On which day does it first reach the top?')
def run(ckpt, label, n=3, temp=0.7):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    print(f'\n{"="*68}\n{label}  ({ckpt})\n{"="*68}')
    with torch.no_grad():
        for k in range(n):
            p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
            plan = m.sample_plan(p_ids, p_attn, temp=temp, sample=True)
            gen = m.generate_answer(p_ids, p_attn, plan, temp=temp, sample=True, max_new_tokens=64)
            print(f'  sample {k+1}  plan: {" → ".join(decode_plan(plan[0]))}')
            print(f'            answer: {m.tok.decode(gen[0], skip_special_tokens=True).strip()}')
    del m; torch.cuda.empty_cache()
print('HARD PROMPT (not in data):\n ', HARD_PROMPT)
run('joint_ckpt', 'SFT only'); run('rl_ckpt', 'SFT + GRPO')
